# Week9 End-toEnd IR Applications


Your task in this activity is to:

- To implement all the steps discussed in the lecture and apply your IR algorithm to the collection of documents and the set of queries provided with this notebook.

Before you begin, attempt the following exercise:

There are a number of stemming algorithms available as part of NLP toolkits (e.g., as part of NLTK), and in practice you do not need to implement your own stemmer or lemmatizer from scratch. However, as with any of the available toolkits implemented for you, it is a good idea to understand their capabilities and limitations.

To practice with different algorithms:

- Revisit Section 3.6 of Natural Language Processing with Python by Bird et al. (http://www.nltk.org/book_1ed/ch03.html).
- Select some input text and run tokenization on it.
- Apply a range of available stemming and lemmatization algorithms from NLTK (for more
information, see https://www.nltk.org/api/nltk.stem.html and https://www.nltk.org/howto/stem.html) and compare the results.

This is an open-ended task.

## Alternative IR Tools for the future

There are many great IR libraries that can be used for IR related tasks. In this lab we are doing a ``barebones'' implementation, but you can look at the following libraries for more complete solutions:
* Anserini (https://github.com/castorini/anserini)
* Pyterrier (https://github.com/terrier-org/pyterrier)
* IR Meaures (https://ir-measur.es/en/latest/)
* Elastic Search (https://www.elastic.co/guide/en/elasticsearch/client/python-api/current/index.html)
* Gensim (https://radimrehurek.com/gensim/)

## Step 1: Read in the data

There are three components to the provided `CISI` dataset:
- documents with their ids and content – there are $1460$ of those to be precise;
- questions / queries with their ids and content – there are $112$ of those;
- mapping between the queries and relevant documents.

First, let's read in documents from the `CISI.ALL` file and store the result in `documents` data structure – set of tuples of document ids matched with contents:

In [ ]:
#if using Google Colab
#from google.colab import drive
#drive.mount('/content/drive')

def read_documents():
    f = open("cisi/CISI.ALL") # if accessed locally
    #f = open("/content/drive/My Drive/" + [<specify location on Drive>] + "cisi/CISI.ALL") #if via Colab
    merged = ""

    for a_line in f.readlines():
        if a_line.startswith("."):
            merged += "\n" + a_line.strip()
        else:
            merged += " " + a_line.strip()

    documents = {}

    content = ""
    doc_id = ""

    for a_line in merged.split("\n"):
        if a_line.startswith(".I"):
            doc_id = a_line.split(" ")[1].strip()
        elif a_line.startswith(".X"):
            documents[doc_id] = content
            content = ""
            doc_id = ""
        else:
            content += a_line.strip()[3:] + " "
    f.close()
    return documents

documents = read_documents()
print(f"{len(documents)} documents in total")
print("Document with id 1:")
print(documents.get("1"))

Second, let's read in queries from the `CISI.QRY` file and store the result in `queries` data structure – set of tuples of query ids matched with contents:

In [ ]:
def read_queries():
    f = open("cisi/CISI.QRY") # same as above; use a different command if running in Colab
    merged = ""

    for a_line in f.readlines():
        if a_line.startswith("."):
            merged += "\n" + a_line.strip()
        else:
            merged += " " + a_line.strip()

    queries = {}

    content = ""
    qry_id = ""

    for a_line in merged.split("\n"):
        if a_line.startswith(".I"):
            if not content=="":
                queries[qry_id] = content
                content = ""
                qry_id = ""
            qry_id = a_line.split(" ")[1].strip()
        elif a_line.startswith(".W") or a_line.startswith(".T"):
            content += a_line.strip()[3:] + " "
    queries[qry_id] = content
    f.close()
    return queries

queries = read_queries()
print(f"{len(queries)} queries in total")
print("Query with id 1:")
print(queries.get("1"))

Finally, let's read in the mapping between the queries and the documents – we'll keep these in the `mappings` data structure – with tuples where each query index (key) corresponds to the list of one or more document indices (value):

In [ ]:
def read_mappings():
    f = open("cisi/CISI.REL") # different command if running in Colab; see above

    mappings = {}

    for a_line in f.readlines():
        voc = a_line.strip().split()
        key = voc[0].strip()
        current_value = voc[1].strip()
        value = []
        if key in mappings.keys():
            value = mappings.get(key)
        value.append(current_value)
        mappings[key] = value

    f.close()
    return mappings

mappings = read_mappings()
print(f"{len(mappings)} mappings in total")
print(mappings.keys())
print("Mapping for query with id 1:")
print(mappings.get("1"))

## Step 2: Preprocess the data

Pratise application of the following steps:
- tokenize the texts
- put all to lowercase
- remove stopwords
- apply stemming

Implement and apply these steps to a sample text:

In [ ]:
import nltk
import string

nltk.download('stopwords')
nltk.download('punkt')

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.lancaster import LancasterStemmer

def process(text):
    stoplist = set(stopwords.words('english'))
    st = LancasterStemmer()
    # TODO
    word_list =  [st.stem(word) for word in ...
                 # a tokenized list of words, all converted to lowercase,
                 # if the word is not in the stoplist and not a punctuation mark (from string.punctuation)
                 ]
    return word_list

word_list = process(documents.get("27"))
print(word_list)

## Step 3: Term weighing

First calculate the term frequency in each document:

In [ ]:
def get_terms(text):
    terms = {}
    st = LancasterStemmer()
    # TODO
    stoplist = # as above
    word_list = # as above
    for word in word_list:
        terms[word] = terms.get(word, 0) + 1
    return terms

doc_terms = {}
qry_terms = {}
for doc_id in documents.keys():
    # TODO
    doc_terms[doc_id] = get_terms(# Apply to the content of the document with id doc_id
                                  )
for qry_id in queries.keys():
    # TODO
    qry_terms[qry_id] = get_terms(# Apply to the content of the query with id qry_id
                                  )


print(f"{len(doc_terms)} documents in total") # Sanity check – this should be the same number as before
d1_terms = doc_terms.get("1")
print("Terms and frequencies for document with id 1:")
print(d1_terms)
print(f"{len(d1_terms)} terms in this document")
print()
print(f"{len(qry_terms)} queries in total") # Sanity check – this should be the same number as before
q1_terms = qry_terms.get("1")
print("Terms and frequencies for query with id 1:")
print(q1_terms)
print(f"{len(q1_terms)} terms in this query")

Second, collect shared vocabulary from all documents and queries:

In [ ]:
def collect_vocabulary():
    all_terms = []
    for doc_id in doc_terms.keys():
        for term in doc_terms.get(doc_id).keys():
            all_terms.append(term)
    for qry_id in qry_terms.keys():
        # TODO: YOUR CODE HERE. Apply the same procedure to the query terms
    return sorted(set(all_terms))

all_terms = collect_vocabulary()
print(f"{len(all_terms)} terms in the shared vocabulary") # This should be the same number as before
print("First 10:")
print(all_terms[:10])

Represent each document and query as vectors containing word counts in the shared space:

In [ ]:
def vectorize(input_terms, shared_vocabulary):
    output = {}
    for item_id in input_terms.keys(): # e.g., a document in doc_terms
        terms = input_terms.get(item_id)
        output_vector = []
        for word in shared_vocabulary:
            if word in terms.keys():
                # TODO: YOUR CODE GOES HERE
                # Add the raw count of the word from the shared vocabulary in doc to the doc vector
                output_vector.append(int(terms.get(word)))
            else:
                # TODO
                # If the word from the shared vocabulary is not in doc, add 0 to the doc vector in this position
                output_vector.append(0)
        output[item_id] = output_vector
    return output

# TODO
doc_vectors = vectorize(# Apply vectorize to the doc_terms and the shared vocabulary all_terms
                        )

# TODO
qry_vectors = vectorize(# Apply vectorize to the qry_terms and the shared vocabulary all_terms
                        )

print(f"{len(doc_vectors)} document vectors") # This should be the same number as before
d1460_vector = doc_vectors.get("1460")
print(f"{len(d1460_vector)} terms in this document") # This should be the same number as before
print(f"{len(qry_vectors)} query vectors") # This should be the same number as before
q112_vector = qry_vectors.get("112")
print(f"{len(q112_vector)} terms in this query") # This should be the same number as before

In [ ]:
import math

def calculate_idfs(shared_vocabulary, d_terms):
    doc_idfs = {}
    for term in shared_vocabulary:
        doc_count = 0 # the number of documents containing this term
        for doc_id in d_terms.keys():
            terms = d_terms.get(doc_id)
            if term in terms.keys():
                doc_count += 1
        doc_idfs[term] = math.log(float(len(d_terms.keys()))/float(1 + doc_count), 10)
    return doc_idfs

# TODO: YOUR CODE GOES HERE
doc_idfs = calculate_idfs(# Apply calculate_idfs to the shared vocabulary all_terms and to doc_terms
                        )
print(f"{len(doc_idfs)} terms with idf scores") # This should be the same number as before
print("Idf score for the word system:")
print(doc_idfs.get("system"))

In [ ]:
def vectorize_idf(input_terms, input_idfs, shared_vocabulary):
    output = {}
    for item_id in input_terms.keys():
        # TODO
        terms = # Collect terms from the document
        output_vector = []
        for term in shared_vocabulary:
            if term in terms.keys():
                output_vector.append(input_idfs.get(term)*float(terms.get(term)))
            else:
                output_vector.append(float(0))
        output[item_id] = output_vector
    return output

# TODO: YOUR CODE GOES HERE
doc_vectors = vectorize_idf(# apply to the relevant data structures
                            )

print(f"{len(doc_vectors)} document vectors") # This should be the same number as before
print("Number of idf-scored words in a particular document:")
print(len(doc_vectors.get("1460"))) # This should be the same number as before

## Step 4: Retrieval of the most similar documents

Use cosine similarity on a toy example:

In [ ]:
import math

query = [1, 1]
document = [3, 5]

def length(vector):
    sq_length = 0
    # TODO
    # Calculate vector length
    return math.sqrt(sq_length)

def dot_product(vector1, vector2):
    if len(vector1)==len(vector2):
        dot_prod = 0
         # TODO
        # Calculate dot product
        return dot_prod
    else:
        return "Unmatching dimensionality"

# TODO: YOUR CODE 
def calculate_cosine(query, document):
    cosine =  # Compute dot product / lengths of vectors multiplied
    return cosine

cosine = calculate_cosine(query, document)
print (cosine) # the result should be ~0.97

Get cosine similarity for some examples of a particular query and a particular document from the collection:

In [ ]:
document = doc_vectors.get("60")
query = qry_vectors.get("3")

#TODO
cosine =  # as above
print(cosine) # the result should be ~0.22

Apply the search algorithm to find relevant documents for a particular query:

In [ ]:
from operator import itemgetter

results = {}

for doc_id in doc_vectors.keys():
    document = doc_vectors.get(doc_id)
    cosine = calculate_cosine(query, document)
    results[doc_id] = cosine

# TODO
for items in # print out the top 10 most similar documents according to consine similarity
    print(f"Doc {items[0]} with similarity {round(items[1], 4)}")